In [ ]:
#  Scenario 1: AI Customer Support Multi-Agent System
# 🎯 Objective:
# Build a multi-agent customer support system that can classify, delegate, and resolve user queries.

# 💼 Problem Statement
# A company wants to automate its support system using AI agents.

# The system should:

# Understand customer queries
# Route them to the correct department
# Provide responses
# 👥 Agents to Build
# Classifier Agent
# Identifies query type (Billing / Technical / General)
# Billing Agent
# Handles payment/refund queries
# Technical Support Agent
# Handles app/software issues
# Response Agent
# Combines outputs into a final response
# ⚙️ Tasks to Implement
# Multi-agent architecture (at least 3 agents mandatory)
# Task delegation logic
# Basic agent communication (message passing / function calls)
# 💡 Sample Inputs
# “I was charged twice for my subscription”
# “The app crashes when I open it”
# “What are your pricing plans?”
# 🧪 Expected Output
# Correct classification of query
# Delegation to appropriate agent
# Final structured response
# 🧰 Tools (Choose Any)
# Python (basic functions or LLM APIs)
# LangGraph (workflow)
# CrewAI (role-based agents)
# AutoGen (agent conversation)
# Microsoft Copilot Studio (optional low-code)
# 📦 Deliverables
# Working code / flow
# Architecture diagram
# 2–3 test case outputs
# ⏱️ Suggested Time Split
# Design: 20 min
# Implementation: 70 min
# Testing & documentation: 30 min

In [ ]:
# Architecturee------->>>>

# User Query
#    ↓
# LLM Classifier Agent
#    ↓
# Task Delegation Logic
#    ↓
# Specialized LLM Agents
#    ↓
# Response Agent
#    ↓
# Final Output

In [22]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL_NAME = "llama-3.3-70b-versatile"

In [23]:
def classifier_agent(query):
    try:
        prompt = f"""
        Classify the following query into:
        billing, technical, or general.

        Only return one word.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip().lower()

    except Exception as e:
        print("Error in classifier:", e)
        return "general"   # fallback

In [24]:
def billing_agent(query):
    try:
        prompt = f"""
        You are a billing support assistant.

        Respond politely to the user's billing issue.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",   # ✅ FIXED
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        return {
            "department": "Billing",
            "message": response.choices[0].message.content.strip()
        }

    except Exception as e:
        print("Billing error:", e)
        return {
            "department": "Billing",
            "message": "We are facing issues. Please try again later."
        }

In [25]:
def technical_agent(query):
    try:
        prompt = f"""
        You are a technical support assistant.

        Help resolve the issue step-by-step.

        Query: {query}
        """

        response = client.chat.completions.create(
            model=MODEL_NAME,   # ✅ FIXED
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        return {
            "department": "Technical Support",
            "message": response.choices[0].message.content.strip()
        }

    except Exception as e:
        print("Technical error:", e)
        return {
            "department": "Technical Support",
            "message": "Please try reinstalling or updating the app."
        }

In [26]:
def general_agent(query):
    prompt = f"""
    You are a customer support assistant.

    Answer general queries clearly.

    Query: {query}
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "department": "General Inquiry",
        "message": response.choices[0].message.content.strip()
    }

In [38]:
# def response_agent(classification, response):
#     return {
#         "query_type": classification,
#         "department": response["department"],
#         "final_response": response["message"]
#     }


def response_agent(classification, response):
    return f"""
==============================
📌 CUSTOMER SUPPORT RESPONSE
==============================

🔍 Query Type   : {classification.upper()}
🏢 Department   : {response['department']}

💬 Response:
{response['message']}

==============================
"""

In [39]:
def multi_agent_system(query):
    print(f"\nUser Query: {query}")

    # Step 1: Classification
    category = classifier_agent(query)
    print("Classified as:", category)

    # Step 2: Delegation
    if category == "billing":
        result = billing_agent(query)
    elif category == "technical":
        result = technical_agent(query)
    else:
        result = general_agent(query)

    # Step 3: Final Response
    final_output = response_agent(category, result)

    return final_output

In [40]:
queries = [
    "I was charged twice for my subscription",
    "The app crashes when I open it",
    "What are your pricing plans?"
]


for q in queries:
    output = multi_agent_system(q)
    print("Final Output:", output)


User Query: I was charged twice for my subscription
Classified as: billing
Final Output: 
📌 CUSTOMER SUPPORT RESPONSE

🔍 Query Type   : BILLING
🏢 Department   : Billing

💬 Response:
I'm so sorry to hear that you were charged twice for your subscription. I'm here to help you resolve this issue as quickly as possible.

Can you please provide me with more details about the duplicate charge, such as the date of the charges and the amount that was deducted from your account? Additionally, could you please confirm your subscription details, including your account name and the type of subscription you have with us?

I'll do my best to investigate this matter and ensure that the duplicate charge is refunded to you promptly. Your satisfaction is our top priority, and I appreciate your patience and cooperation in this matter.



User Query: The app crashes when I open it
Classified as: technical
Final Output: 
📌 CUSTOMER SUPPORT RESPONSE

🔍 Query Type   : TECHNICAL
🏢 Department   : Technical Su

In [41]:
queries = [
    "My payment failed but money was deducted",
    "I am getting an error while logging in",
    "Do you offer any free trial?"
]


for q in queries:
    output = multi_agent_system(q)
    print("Final Output:", output)


User Query: My payment failed but money was deducted
Classified as: billing
Final Output: 
📌 CUSTOMER SUPPORT RESPONSE

🔍 Query Type   : BILLING
🏢 Department   : Billing

💬 Response:
I'm so sorry to hear that your payment failed, but the amount was still deducted from your account. I can imagine how frustrating that must be for you.

Can you please provide me with more details about the issue, such as the date of the payment attempt, the amount that was deducted, and the error message you received (if any)? This will help me to investigate the matter further and assist you in resolving the issue as quickly as possible.

Additionally, I would like to request your account information, such as your account number or the email address associated with your account, so that I can look into this matter more efficiently.

Please be assured that I'm here to help, and I'll do my best to resolve this issue for you. If the payment was indeed deducted in error, I'll work with our team to process a

In [42]:
queries = [
    "My account is not accessible after payment",
    "How does your platform work?",
    "App crashes after I upgraded my plan"
]


for q in queries:
    output = multi_agent_system(q)
    print("Final Output:", output)


User Query: My account is not accessible after payment
Classified as: technical
Final Output: 
📌 CUSTOMER SUPPORT RESPONSE

🔍 Query Type   : TECHNICAL
🏢 Department   : Technical Support

💬 Response:
I'm here to help you resolve the issue with your account accessibility after payment. Let's go through the steps to troubleshoot and fix the problem.

**Step 1: Confirm Payment Status**
Can you please confirm that the payment has been successfully processed? Check your email for a payment confirmation or log in to your payment method (e.g., PayPal, credit card) to verify the transaction.

**Step 2: Check Account Status**
Try to log in to your account using the same credentials you used before. If you're unable to log in, please check if you've received any error messages or notifications. Are you seeing any specific error messages, such as "Account locked" or "Invalid credentials"?

**Step 3: Verify Account Information**
Double-check that your account information, including your username a